# Here are the library you need to import

In [1]:
import PIL
import time
import torch
import torchvision
import torch.nn.functional as F
from einops import rearrange
from torch import nn
import torch.nn.init as init
from torch.utils.data import Dataset, DataLoader
import glob
import scipy.io
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
import numpy as np
from random import randint
import random
import time
import re
import itertools
from timm.models.layers import DropPath
from einops import rearrange
from scipy import ndimage
from skimage import io
from skimage import transform
from natsort import natsorted
from skimage.transform import rotate, AffineTransform
from timm.models.layers import DropPath, to_3tuple, trunc_normal_
from monai.transforms import (
    AsDiscrete,
    AddChanneld,
    Compose,
    CropForegroundd,
    LoadImaged,
    Orientationd,
    RandFlipd,
    RandCropByPosNegLabeld,
    RandShiftIntensityd,
    ScaleIntensityRanged,
    Spacingd,
    RandRotate90d,
    ToTensord,
    RandAffined,
    RandCropByLabelClassesd,
    SpatialPadd,
    RandAdjustContrastd,
    RandShiftIntensityd,
    ScaleIntensityd,
    NormalizeIntensityd,
    RandScaleIntensityd,
    RandGaussianNoised,
    RandGaussianSmoothd,
    ScaleIntensityRangePercentilesd,
    Resized,
    Transposed,
    RandSpatialCropd,
    RandSpatialCropSamplesd,
    ResizeWithPadOrCropd
)
from monai.transforms import (CastToTyped,
                              Compose, CropForegroundd, EnsureChannelFirstd, LoadImaged,
                              NormalizeIntensity, RandCropByPosNegLabeld,
                              RandFlipd, RandGaussianNoised,
                              RandGaussianSmoothd, RandScaleIntensityd,
                              RandZoomd, SpatialCrop, SpatialPadd, EnsureTyped)
from monai.transforms.compose import MapTransform
from monai.config import print_config
from monai.metrics import DiceMetric
from skimage.transform import resize
import scipy.io
import matplotlib.pyplot as plt

import numpy as np

import torch
from torch import nn, einsum
import torch.nn.functional as F

import copy
from diffusion.Create_diffusion import *
from diffusion.resampler import *
import nibabel as nib
from torch.utils.tensorboard import SummaryWriter

# Build the data loader using the monai library

In [2]:
# ── Config: Brain / SynthRAD2023 ─────────────────────────────────────────────

BATCH_SIZE_TRAIN = 4                 
img_size         = (192, 192, 96)     # changed from (256,256,128) → brain crop size
patch_size       = (64, 64, 4)        # unchanged — matches original brain paper value
spacing          = (1, 1, 1)          # changed from (2,2,2) → SynthRAD brain is 1x1x1 mm³
patch_num        = 2                  # unchanged
channels         = 1                  # unchanged
metric           = torch.nn.L1Loss()  # unchanged

# ── Added: paths and training settings ───────────────────────────────────────
DATA_ROOT = '/home/teaching/ak_dl/brain_npy'
SAVE_DIR         = './checkpoints_brain'
LOG_DIR          = './runs'
CT_CLIP          = (-1024, 1650)      # HU clip range from paper
LR               = 3e-5              # brain learning rate from paper
WEIGHT_DECAY     = 1e-5
EPOCHS           = 500
VIS_EVERY        = 10                 # log sample images to TensorBoard every N epochs

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'img_size  : {img_size}')
print(f'patch_size: {patch_size}')
print(f'spacing   : {spacing}')
print(f'data root : {DATA_ROOT}')

img_size  : (192, 192, 96)
patch_size: (64, 64, 4)
spacing   : (1, 1, 1)
data root : /home/teaching/ak_dl/brain_npy


# Data processing for mat files (contain {'image', 'label'}) (for nii files, in the next block)

In [3]:
class CustomDataset(Dataset):
    def __init__(self, imgs_path, labels_path=None, train_flag=True):
        self.train_flag = train_flag
        self.files = natsorted(glob.glob(imgs_path + "*.npy"),
                               key=lambda y: y.lower())
        if len(self.files) == 0:
            raise FileNotFoundError(f"No .npy files found in {imgs_path}")
        print(f"Found {len(self.files)} preprocessed volumes "
              f"[{'train' if train_flag else 'val/test'}]")

        self.patch_transform = Compose([
            RandSpatialCropSamplesd(
                keys=["image", "label"],
                roi_size=patch_size,
                num_samples=patch_num,
                random_size=False,
            ),
            ToTensord(keys=["image", "label"]),
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data   = np.load(self.files[idx])
        mr_vol = data[0]   # MRI  [-1, 1]
        ct_vol = data[1]   # CT   [-1, 1]

        data_dict = {
            "image": mr_vol[np.newaxis],
            "label": ct_vol[np.newaxis],
        }

        if not self.train_flag:
            img_tensor   = torch.from_numpy(mr_vol[np.newaxis]).float()
            label_tensor = torch.from_numpy(ct_vol[np.newaxis]).float()
        else:
            out   = self.patch_transform(data_dict)
            img   = np.zeros([patch_num, patch_size[0], patch_size[1], patch_size[2]])
            label = np.zeros([patch_num, patch_size[0], patch_size[1], patch_size[2]])
            for i, sample in enumerate(out):
                img[i]   = sample["image"].numpy()
                label[i] = sample["label"].numpy()
            img_tensor   = torch.unsqueeze(torch.from_numpy(img.copy()),   1).float()
            label_tensor = torch.unsqueeze(torch.from_numpy(label.copy()), 1).float()

        return img_tensor, label_tensor

# Data processing for nii files (including reading, adding channels, align orientation, align spacing, normalization (MRI and CT  has different normalization), cropping and padding, and finally extracing patches for training.

# But trust me, this process takes a lot of time. Try to process all the data before you run the model instead of processing them on-the-fly. At least try to get rid of the spacing and orientation.

In [5]:
# # Only be careful about the ResizeWithPadOrCropd. I am not sure should you use it or not. In my case,
# # I need a volume with fixed size.

# # One more thing, be careful of the normalization, CT is quantative and MRI is not, so they need different normalization here.
# # Maybe not your case.

# class CustomDataset(Dataset):
#     def __init__(self,imgs_path,labels_path, train_flag = True):
#         self.imgs_path = imgs_path
#         self.labels_path = labels_path
#         self.train_flag = train_flag
#         file_list = natsorted(glob.glob(self.imgs_path + "*nii.gz"), key=lambda y: y.lower())     
#         label_list = natsorted(glob.glob(self.labels_path + "*nii.gz"), key=lambda y: y.lower())
#         self.data = []
#         self.label = []
#         for img_path in file_list:
#             class_name = img_path.split("/")[-1]
#             self.data.append([img_path, class_name])
#         for label_path in label_list:
#                 class_name = label_path.split("/")[-1]
#                 self.label.append([label_path, class_name])
#         self.train_transforms = Compose(
#                 [
#                     LoadImaged(keys=["image","label"],reader='nibabelreader'),
#                     AddChanneld(keys=["image","label"]),
#                     Orientationd(keys=["image","label"], axcodes="RAS"),
#                     Spacingd(
#                         keys=["image","label"],
#                         pixdim=spacing,
#                         mode=("bilinear"),
#                     ),
#                     ScaleIntensityd(keys=["image"], minv=-1, maxv=1.0),
#                     ScaleIntensityRanged(
#                         keys=["label"],
#                         a_min=0,
#                         a_max=2674,
#                         b_min=-1,
#                         b_max=1.0,
#                         clip=True,
#                     ),
# #                     ResizeWithPadOrCropd(
# #                           keys=["image","label"],
# #                           spatial_size=img_size,
# #                           constant_values = -1,
# #                     ),
#                     RandSpatialCropSamplesd(keys=["image","label"],
#                                       roi_size = patch_size,
#                                       num_samples = patch_num,
#                                       random_size=False,
#                                       ),

#                     ToTensord(keys=["image","label"]),
#                 ]
#             )
#         self.test_transforms = Compose(
#                 [
#                     LoadImaged(keys=["image","label"],reader='nibabelreader'),
#                     AddChanneld(keys=["image","label"]),
#                     Orientationd(keys=["image","label"], axcodes="RAS"),
#                     Spacingd(
#                         keys=["image","label"],
#                         pixdim=spacing,
#                         mode=("bilinear"),
#                     ),
#                     ScaleIntensityd(keys=["image"], minv=-1, maxv=1.0),
#                     ScaleIntensityRanged(
#                         keys=["label"],
#                         a_min=0,
#                         a_max=2674,
#                         b_min=-1,
#                         b_max=1.0,
#                         clip=True,
#                     ),
                    
# #                     ResizeWithPadOrCropd(
# #                           keys=["image","label"],
# #                           spatial_size=img_size,
# #                           constant_values = -1,
# #                     ),
#                     ToTensord(keys=["image","label"]),
#                 ]
#             ) 
#     def __len__(self):
#         return len(self.data)
    
#     def __getitem__(self, idx):

#         img_path, class_name = self.data[idx]
#         label_path, class_name = self.label[idx]
#         cao = {"image":img_path,'label':label_path}

#         if not self.train_flag:
#             affined_data_dict = self.test_transforms(cao)   
#             img_tensor = affined_data_dict['image'].to(torch.float)
#             label_tensor = affined_data_dict['label'].to(torch.float)
#         else:
#             affined_data_dict = self.train_transforms(cao)   
#             img = np.zeros([patch_num, patch_size[0], patch_size[1], patch_size[2]])
#             label = np.zeros([patch_num, patch_size[0], patch_size[1], patch_size[2]])
#             for i,after_l in enumerate(affined_data_dict):
#                 img[i,:,:,:] = after_l['image']
#                 label[i,:,:,:] = after_l['label']
#             img_tensor = torch.unsqueeze(torch.from_numpy(img.copy()), 1).to(torch.float)
#             label_tensor = torch.unsqueeze(torch.from_numpy(label.copy()), 1).to(torch.float)
        
#         return img_tensor,label_tensor

# Build the MC-IDDPM process

In [4]:
diffusion_steps=1000
learn_sigma=True
timestep_respacing=[50]

sigma_small=False
class_cond=False
noise_schedule='linear'
use_kl=False
predict_xstart=False        # ← changed from True to False
rescale_timesteps=True
rescale_learned_sigmas=True
use_checkpoint=False

diffusion = create_gaussian_diffusion(
    steps=diffusion_steps,
    learn_sigma=learn_sigma,
    sigma_small=sigma_small,
    noise_schedule=noise_schedule,
    use_kl=use_kl,
    predict_xstart=predict_xstart,
    rescale_timesteps=rescale_timesteps,
    rescale_learned_sigmas=rescale_learned_sigmas,
    timestep_respacing=timestep_respacing,
)
schedule_sampler = UniformSampler(diffusion)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda:0


# Build the MC-IDDPM network

In [5]:
# Here enter your network parameters:num_channels means the initial channels in each block,
# channel_mult means the multipliers of the channels (in this case, 128,128,256,256,512,512 for the first to the sixth block),
# attention_resulution means we use the transformer blocks in the third to the sixth block
# number of heads, window size in each transformer block
# 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

num_channels=64
attention_resolutions="32,16,8"
channel_mult = (1, 2, 3, 4)
num_heads=[4,4,8,16]
window_size = [[4,4,4],[4,4,4],[4,4,2],[4,4,2]]
num_res_blocks = [2,2,2,2]
sample_kernel=([2,2,2],[2,2,1],[2,2,1],[2,2,1]),

attention_ds = []
for res in attention_resolutions.split(","):
    attention_ds.append(int(res))
class_cond = False
use_scale_shift_norm=True
resblock_updown = False
dropout = 0

from network.Diffusion_model_transformer import *
A_to_B_model = SwinVITModel(
          image_size=patch_size,
          in_channels=2,
          model_channels=num_channels,
          out_channels=2,
          dims=3,
          sample_kernel = sample_kernel,
          num_res_blocks=num_res_blocks,
          attention_resolutions=tuple(attention_ds),
          dropout=dropout,
          channel_mult=channel_mult,
          num_classes=None,
          use_checkpoint=False,
          use_fp16=False,
          num_heads=num_heads,
          window_size = window_size,
          num_head_channels=64,
          num_heads_upsample=-1,
          use_scale_shift_norm=use_scale_shift_norm,
          resblock_updown=resblock_updown,
          use_new_attention_order=False,
      ).to(device)

cuda:0


In [6]:
# #In case you want to use CNN
# from network.Diffusion_model_Unet import *
# A_to_B_model = UNetModel(
#         img_size = patch_size,
#         image_size=patch_size[0],
#         in_channels=2,
#         model_channels=num_channels,
#         out_channels=2,
#         dims = 3,
#         num_res_blocks=num_res_blocks[0],
#         attention_resolutions=tuple(attention_ds),
#         dropout=0.,
#         sample_kernel=sample_kernel,
#         channel_mult=channel_mult,
#         num_classes=(128 if class_cond else None),
#         use_checkpoint=False,
#         use_fp16=False,
#         num_heads=4,
#         num_head_channels=64,
#         num_heads_upsample=-1,
#         use_scale_shift_norm=use_scale_shift_norm,
#         resblock_updown=False,
#         use_new_attention_order=False,
#     ).to(device)

# Call the optimizer and ready for start

In [6]:
pytorch_total_params = sum(p.numel() for p in A_to_B_model.parameters())
print('parameter number is ' + str(pytorch_total_params))
torch.backends.cudnn.benchmark = True
optimizer = torch.optim.AdamW(A_to_B_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler()

parameter number is 54438954


# Build the training function. Run the training function once = one epoch

In [7]:
def train(model, optimizer, data_loader1, loss_history, epoch, writer, global_step):

    model.train()
    total_samples = len(data_loader1.dataset)
    A_to_B_loss_sum = []
    total_time = 0

    for i, (x1, y1) in enumerate(data_loader1):

        traintarget   = y1.view(-1, 1, patch_size[0], patch_size[1], patch_size[2]).to(device)
        traincondition = x1.view(-1, 1, patch_size[0], patch_size[1], patch_size[2]).to(device)

        t, weights = schedule_sampler.sample(traincondition.shape[0], device)

        aa = time.time()

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            all_loss    = diffusion.training_losses(A_to_B_model, traintarget, traincondition, t)
            A_to_B_loss = (all_loss["loss"] * weights).mean()
            A_to_B_loss_sum.append(all_loss["loss"].mean().detach().cpu().numpy())

        scaler.scale(A_to_B_loss).backward()

        # Unscale before clipping so grad norm is in real units
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        # ── TensorBoard: per-step logging ─────────────────────────────────
        writer.add_scalar("Loss/train_step",     A_to_B_loss.item(), global_step)
        writer.add_scalar("GradNorm/train_step", grad_norm,          global_step)
        global_step += 1

        total_time += time.time() - aa
        if i % 30 == 0:
            print(f'optimization time: {time.time()-aa:.3f}s')
            print(f'[{i * BATCH_SIZE_TRAIN:5}/{total_samples:5}'
                  f' ({100 * i / len(data_loader1):3.0f}%)]'
                  f'  A_to_B_Loss: {np.nanmean(A_to_B_loss_sum):6.7f}')

    average_loss = np.nanmean(A_to_B_loss_sum)
    loss_history.append(average_loss)

    # ── TensorBoard: per-epoch logging ────────────────────────────────────
    writer.add_scalar("Loss/train_epoch", average_loss, epoch)

    print(f"Total time per sample: {total_time:.3f}s")
    print(f"Averaged loss: {average_loss}")
    return average_loss, global_step

# Build the testing function.

In [8]:
def evaluate(model, epoch, path, data_loader1, best_loss, writer):
    model.eval()
    prediction = []
    true = []
    img = []
    loss_all = []
    aa = time.time()
    with torch.no_grad():
        for i, (x1, y1) in enumerate(data_loader1):
                # target is the target CT
                # condition is the input MRI
                # sampled_images is the synthetic CT
                target = y1.to(device)            
                condition = x1.to(device)
                with torch.cuda.amp.autocast():
                      sampled_images = inferer(condition, diffusion_sampling, model)
                loss = metric(sampled_images, target)
                print('L1 loss: '+ str(loss))
                img.append(x1.cpu().numpy())
                true.append(target.cpu().numpy())
                prediction.append(sampled_images.cpu().numpy())    
                loss_all.append(loss.item())        # ← changed from loss.cpu().numpy()
        
        print('optimization time: '+ str(1*(time.time()-aa)))  

        # ── Save as .nii.gz instead of .mat ───────────────────────────────
        lo, hi = CT_CLIP
        for i, sct_np in enumerate(prediction):
            sct_hu = (sct_np[0, 0] + 1.0) / 2.0 * (hi - lo) + lo
            nib.save(
                nib.Nifti1Image(sct_hu, np.eye(4)),
                path + 'sct_epoch' + str(epoch) + '_sample' + str(i) + '.nii.gz'
            )
        if np.mean(loss_all) < best_loss:
            for i, sct_np in enumerate(prediction):
                sct_hu = (sct_np[0, 0] + 1.0) / 2.0 * (hi - lo) + lo
                nib.save(
                    nib.Nifti1Image(sct_hu, np.eye(4)),
                    path + 'best_sct_sample' + str(i) + '.nii.gz'
                )

        # ── TensorBoard ───────────────────────────────────────────────────
        writer.add_scalar("Loss/val_epoch", np.mean(loss_all), epoch)

        # Log middle slice of first sample
        mid    = prediction[0].shape[-1] // 2
        def norm01(x):
            return (x - x.min()) / (x.max() - x.min() + 1e-8)
        writer.add_image("Val/MRI_input",      norm01(img[0][0, 0, :, :, mid])[None],        epoch)
        writer.add_image("Val/CT_groundtruth", norm01(true[0][0, 0, :, :, mid])[None],       epoch)
        writer.add_image("Val/CT_synthetic",   norm01(prediction[0][0, 0, :, :, mid])[None], epoch)

        return np.mean(loss_all)

# Start the training and testing

In [9]:
# ── Dataloaders ───────────────────────────────────────────────────────────────
training_set1 = CustomDataset('/home/teaching/ak_dl/brain_npy/imagesTr/', train_flag=True)
testing_set1  = CustomDataset('/home/teaching/ak_dl/brain_npy/imagesTs/', train_flag=False)
val_set1      = CustomDataset('/home/teaching/ak_dl/brain_npy/imagesVal/', train_flag=False)

train_params = {'batch_size': BATCH_SIZE_TRAIN, 'shuffle': True,  'pin_memory': False, 'drop_last': False, 'num_workers': 0}
test_params  = {'batch_size': 1,                'shuffle': False, 'pin_memory': False, 'drop_last': False, 'num_workers': 0}

train_loader1 = torch.utils.data.DataLoader(training_set1, **train_params)
test_loader1  = torch.utils.data.DataLoader(testing_set1,  **test_params)
val_loader1   = torch.utils.data.DataLoader(val_set1,      **test_params)

print('Dataloaders ready')

Found 125 preprocessed volumes [train]
Found 37 preprocessed volumes [val/test]
Found 18 preprocessed volumes [val/test]
Dataloaders ready


In [10]:
# Test one single batch
x1, y1 = next(iter(train_loader1))
print(f'MRI batch shape : {x1.shape}')
print(f'CT  batch shape : {y1.shape}')

MRI batch shape : torch.Size([4, 2, 1, 64, 64, 4])
CT  batch shape : torch.Size([4, 2, 1, 64, 64, 4])


In [12]:
# Test one single training step
A_to_B_model.train()
x1, y1 = next(iter(train_loader1))
traintarget    = y1.view(-1, 1, patch_size[0], patch_size[1], patch_size[2]).to(device)
traincondition = x1.view(-1, 1, patch_size[0], patch_size[1], patch_size[2]).to(device)
t, weights = schedule_sampler.sample(traincondition.shape[0], device)
with torch.cuda.amp.autocast():
    all_loss    = diffusion.training_losses(A_to_B_model, traintarget, traincondition, t)
    A_to_B_loss = (all_loss["loss"] * weights).mean()
print(f'Loss: {A_to_B_loss.item()}')
print('Single step OK')

/home/teaching/miniconda3/envs/DL/lib/python3.8/site-packages/torch/nn/functional.py:3657: UserWarning: The default behavior for interpolate/upsample with float scale_factor changed in 1.6.0 to align with other frameworks/libraries, and now uses scale_factor directly, instead of relying on the computed output size. If you wish to restore the old behavior, please set recompute_scale_factor=True. See the documentation of nn.Upsample for details. 
  warnings.warn(


Loss: 0.8100631833076477
Single step OK


In [11]:
from torch.utils.tensorboard import SummaryWriter
import time

run_name = f"brain_{time.strftime('%Y%m%d_%H%M%S')}"
writer   = SummaryWriter(log_dir=os.path.join(LOG_DIR, run_name))

best_loss          = 1
global_step        = 0
train_loss_history = []

for epoch in range(0, 2):
    print('Epoch:', epoch)
    A_to_B_model.train()
    for i, (x1, y1) in enumerate(train_loader1):
        traintarget    = y1.view(-1, 1, patch_size[0], patch_size[1], patch_size[2]).to(device)
        traincondition = x1.view(-1, 1, patch_size[0], patch_size[1], patch_size[2]).to(device)
        t, weights = schedule_sampler.sample(traincondition.shape[0], device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            all_loss    = diffusion.training_losses(A_to_B_model, traintarget, traincondition, t)
            A_to_B_loss = (all_loss["loss"] * weights).mean()
        scaler.scale(A_to_B_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(A_to_B_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        # Only scalar logging — no images
        writer.add_scalar("Loss/train_step", A_to_B_loss.item(), global_step)
        global_step += 1
        if i % 30 == 0:
            print(f'  batch {i} loss: {A_to_B_loss.item():.6f}')
    writer.add_scalar("Loss/train_epoch", np.mean(train_loss_history), epoch)
    print(f'Epoch {epoch} done')

writer.close()
print('Test complete')

: 